# FloodDuo — flood mapping inference

Run the released FloodDuo model on a PlanetScope-style tile and get a binary
water/flood map. Weights + a sample tile download automatically when you run
the cells below — no manual setup beyond installing packages.

This uses the **release checkpoint** (trained on all FloodPlanet + UFO data
pooled together), not one of the paper's leave-one-region-out fold models —
see the note on the [Hugging Face model card](https://huggingface.co/Sk-21/FloodDuo)
for why that distinction matters.

Works on CPU (slower) or GPU; no GPU required for a single tile.

## 1. Install dependencies

In [ ]:
%pip install -q torch torchvision timm rasterio einops huggingface_hub matplotlib numpy

## 2. Download the checkpoint and a sample tile

`floodduo_full.pt` (~1.3 GB) is self-contained: both frozen encoder backbones
(DINOv3-L, Clay v1.5) and FloodDuo's trained adapters/fusion/decoder are all
in this one file, so this is the only download needed to run inference.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root, for `import src`

from huggingface_hub import hf_hub_download

REPO = "Sk-21/FloodDuo"
ckpt_path = hf_hub_download(REPO, "floodduo_full.pt")
sample_image_path = hf_hub_download(REPO, "examples/sample_tile.tif")
sample_mask_path = hf_hub_download(REPO, "examples/sample_mask.tif")
print("downloaded:", ckpt_path)

## 3. Choose your input image

By default this runs on the bundled sample tile (so the rest of the notebook
works with zero changes). To use your own imagery, set `image_path` to your
GeoTIFF and `mask_path = None` (you won't have ground truth to compare
against, and that's fine — the ground-truth panel below is skipped
automatically when `mask_path` is `None`).

Required format: 4-band (B, G, R, NIR), single tile, roughly 512-1024 px per
side — see Section 4 for what happens if it isn't already in reflectance
units.

In [ ]:
image_path = sample_image_path   # <- point this at your own .tif to use your own imagery
mask_path = sample_mask_path     # <- set to None if you don't have a ground-truth mask

## 4. Convert to reflectance (if needed)

The model expects raw surface reflectance (float, roughly 0-0.6). The
FloodPlanet/UFO benchmark tiles (including the bundled sample) are already in
that form. **Your own downloads very likely aren't** — Planet's own Surface
Reflectance product and Sentinel-2 L2A both deliver uint16 GeoTIFFs scaled by
10000 (reflectance x 10000), not pre-scaled floats. Skipping this step on
raw DN input would silently feed the model values in the thousands instead
of fractions, and the prediction would be meaningless.

`to_reflectance` auto-detects which case you're in (integer dtype, or float
values well above 1) and rescales only if needed — it prints what it did.
If your data uses a different scale factor, pass it explicitly, e.g.
`to_reflectance(image, scale=65535)`.

In [ ]:
from src.inference import read_image, to_reflectance

raw_image, profile = read_image(image_path)
image = to_reflectance(raw_image)

## 5. Load the model

In [ ]:
import torch
from src.inference import load_model, predict_array, save_mask

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model(ckpt_path, device=device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"loaded on {device}, {n_params:.0f}M parameters")

## 6. Run inference

In [ ]:
prob, mask = predict_array(model, image, device=device, auto_reflectance=False)  # already converted above
print(f"predicted water fraction: {mask.mean():.1%}")

## 7. Visualize

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt


def false_color(img):
    x = img[[3, 2, 1]].astype(float)  # NIR, R, G
    out = np.empty_like(x)
    for k in range(3):
        lo, hi = np.percentile(x[k], 2), np.percentile(x[k], 98)
        out[k] = np.clip((x[k] - lo) / (hi - lo + 1e-6), 0, 1)
    return np.transpose(out, (1, 2, 0))


gt = None
if mask_path is not None:
    with rasterio.open(mask_path) as src:
        gt = src.read(1) > 0.5

panels = [("False color (NIR-R-G)", false_color(raw_image), None)]
if gt is not None:
    panels.append(("Ground truth", gt, "water"))
panels.append(("FloodDuo prediction", mask, "water"))

fig, ax = plt.subplots(1, len(panels) + 1, figsize=(4.5 * (len(panels) + 1), 5))
for a, (title, data, kind) in zip(ax, panels):
    a.imshow(data, cmap=("Blues" if kind == "water" else None), vmin=0, vmax=1)
    a.set_title(title)
ax[-1].imshow(false_color(raw_image)); ax[-1].imshow(mask, cmap="Blues", alpha=0.45)
ax[-1].set_title("Prediction overlay")
for a in ax: a.set_xticks([]); a.set_yticks([])
fig.tight_layout()

## 8. Save the predicted mask as a GeoTIFF

In [ ]:
out_path = "floodduo_prediction.tif"
save_mask(mask, profile, out_path)
print("saved:", out_path)